In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.13 Fields in Matter: Dielectrics, Polarization, and Magnetic Materials

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.13",
    title="Fields in Matter: Dielectrics, Polarization, and Magnetic " "Materials",
    blurb="The volume's fields finally meet the stuff the world is made "
    "of: bound charges that are really there, a dielectric sphere whose "
    "interior field a numerical solver must reproduce against the exact "
    "answer, the self-consistency loop that predicts a gas's permittivity "
    "from one atom's polarizability, and the mean-field hysteresis at the "
    "heart of every magnet.",
    difficulty="intermediate",
    estimate="130–160 min",
)

## Notebook overview

Twelve notebooks of this volume have treated charges and currents in
vacuum, but almost every field most of us ever meet lives inside
*matter*: the dielectric in a capacitor, the water around an ion, the
iron in a transformer core. Matter responds — its molecules polarize,
its atomic moments align — and the response re-sources the very field
that caused it, a self-consistency that produces some of
electromagnetism's most elegant exact results and, in the magnetic
case, genuine memory. This coda works both chapters of that story
({cite}`griffiths_em` chapters 4 and 6, in the volume's SI
conventions), computationally.

The arc runs from *bookkeeping* to *response*. First bound charges,
which sound like an accounting fiction and are entirely real: we sum
them over a polarized sphere's surface and recover the exact interior
field. Then the classic boundary-value problem of the subject — the
dielectric sphere in a uniform field — solved *numerically* with a
permittivity-jump relaxation solver descended from
[§3.4](laplace-poisson.ipynb), against Griffiths' exact
$3E_0/(\varepsilon_r + 2)$, with the discretization systematics faced
honestly rather than hidden. The Clausius–Mossotti relation then
closes the microscopic loop (one measured atomic polarizability
predicts argon's bulk permittivity to the percent), and the finale
crosses to magnetism, where the same self-consistency run below its
critical temperature stops being a correction and becomes *hysteresis*
— memory in a material, on the mean-field machinery this course builds
for the Ising model in
[§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb).

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**Polarization and bound charge.** When a field displaces the charges
inside molecules, the material acquires a dipole-moment density
$\mathbf P$ (the *polarization*, $\mathrm{C/m^2}$). The net effect of
all those microscopic dipoles is exactly reproduced by *bound* charge
densities

```{math}
:label: eq-fm-bound
\sigma_b \;=\; \mathbf P\cdot\hat{\mathbf n},
\qquad
\rho_b \;=\; -\nabla\cdot\mathbf P ,
```

surface charge where the polarization ends, volume charge where it is
non-uniform. These are not a metaphor: for a uniformly polarized
sphere the $\sigma_b = P\cos\theta$ cap-and-belt pattern produces the
famous *uniform* interior field $\mathbf E = -\mathbf P/(3\varepsilon_0)$,
a result we will obtain by literally summing the surface.

**The auxiliary field and linear dielectrics.** Splitting total charge
into free and bound motivates $\mathbf D = \varepsilon_0\mathbf E +
\mathbf P$, with $\nabla\cdot\mathbf D = \rho_f$: Gauss's law with
only the charge one controls. In a linear dielectric $\mathbf P =
\varepsilon_0\chi_e\mathbf E$, so $\mathbf D = \varepsilon\mathbf E$
with $\varepsilon = \varepsilon_0(1 + \chi_e) \equiv
\varepsilon_0\varepsilon_r$, and the electrostatics of matter becomes
one equation,

```{math}
:label: eq-fm-eps
\nabla\cdot\bigl(\varepsilon\,\nabla V\bigr) \;=\; -\rho_f ,
```

Laplace's equation with a position-dependent coefficient. At an
interface the jump conditions follow: $D_\perp$ continuous (no free
surface charge) and $E_\parallel$ continuous. The showpiece solution is
the dielectric sphere of radius $a$ in a uniform applied field
$E_0\hat{\mathbf z}$: the interior field is *uniform*,

```{math}
:label: eq-fm-sphere
\mathbf E_{\rm in} \;=\; \frac{3}{\varepsilon_r + 2}\,E_0\,
\hat{\mathbf z} ,
```

reduced but not expelled, with the exterior acquiring a perfect dipole
— the electrostatic ancestor of every effective-medium formula.

**Clausius–Mossotti: the local field.** A molecule inside a dielectric
does not feel the macroscopic $\mathbf E$; it sits in a spherical
cavity of its own making, and the field there is $\mathbf E_{\rm loc}
= \mathbf E + \mathbf P/(3\varepsilon_0)$ — the sphere result of
{eq}`eq-fm-sphere` working from the inside out. Demanding
self-consistency between $\mathbf P = n\alpha\mathbf E_{\rm loc}$
(molecular polarizability $\alpha$, number density $n$) and the
macroscopic response gives

```{math}
:label: eq-fm-cm
\frac{\varepsilon_r - 1}{\varepsilon_r + 2}
\;=\; \frac{n\,\alpha}{3\varepsilon_0}
\;=\; \frac{4\pi}{3}\,n\,\alpha' ,
```

with $\alpha' = \alpha/(4\pi\varepsilon_0)$ the tabulated
*polarizability volume*. One atom's response, measured once, prices
the bulk permittivity of the gas — and the relation's divergence at
$4\pi n\alpha'/3 \to 1$ (the "polarization catastrophe") is the
mean-field shadow of ferroelectricity.

**Magnetization, bound currents, and hysteresis.** The magnetic story
runs in parallel: magnetization $\mathbf M$ is reproduced by bound
currents $\mathbf K_b = \mathbf M\times\hat{\mathbf n}$ and
$\mathbf J_b = \nabla\times\mathbf M$, so a uniformly magnetized
cylinder *is* a solenoid with sheet current $K_b = M$, and its on-axis
field is the solenoid formula of [§3.6](magnetostatics.ipynb):

```{math}
:label: eq-fm-cylinder
B_z(0) \;=\; \frac{\mu_0 M}{2}
\left[\frac{z + \ell}{\sqrt{(z+\ell)^2 + R^2}}
- \frac{z - \ell}{\sqrt{(z-\ell)^2 + R^2}}\right]
```

for a cylinder of radius $R$ spanning $|z| \le \ell$. Where the
electric response saturates at Clausius–Mossotti's catastrophe, the
magnetic one — exchange-coupled and orders of magnitude stronger —
routinely crosses it: in the mean-field model each moment feels its
neighbours' average,

```{math}
:label: eq-fm-meanfield
m \;=\; \tanh\!\left(\frac{m + h}{T}\right)
```

(reduced units: $m$ the magnetization per site, $h$ the applied field,
$T$ in units of the coupling; the same equation
[§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)
derives for the Ising ferromagnet). Below $T_c = 1$ it has three
solutions at small $h$, the middle one unstable — and which stable
branch the material occupies depends on where it has *been*. That is
hysteresis: remanence, coercivity, and a loop whose enclosed area is
work dissipated per cycle, all from one transcendental equation.

## Setup

SI conventions as everywhere in Volume III ($\varepsilon_0$, $\mu_0$
explicit in formulas); the numerical exercises work in stated reduced
units where that keeps the arrays tame (each exercise says so). The
permittivity-jump solver is deterministic; nothing here is stochastic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import atm
from scipy.constants import epsilon_0 as EPS0  # F/m
from scipy.constants import k as KB  # J/K

from ecp import validate


def sor_epsilon(eps, V, n_iter, omega=1.95):
    """Red–black SOR relaxation of div(eps grad V) = 0 on a cubic grid.

    The variable-coefficient Laplace solver: each interior site is
    updated to the flux-weighted average of its six neighbours, with the
    face permittivities taken as harmonic means (the discretization that
    keeps D_perp continuous across a jump). Sites are swept in
    checkerboard (red–black) order so over-relaxation converges; the
    boundary values of V are Dirichlet data and are never touched.

    Parameters
    ----------
    eps : numpy.ndarray, shape (N, N, N)
        Relative permittivity on the grid.
    V : numpy.ndarray, shape (N, N, N)
        Potential; modified in place. Boundary layers hold the Dirichlet
        condition.
    n_iter : int
        Number of red–black sweeps (each sweep updates both colours).
    omega : float
        Over-relaxation factor (1 < omega < 2).

    Returns
    -------
    numpy.ndarray
        The relaxed potential (same array as ``V``).
    """

    def hmean(e1, e2):
        return 2.0 * e1 * e2 / (e1 + e2)

    c = eps[1:-1, 1:-1, 1:-1]
    f = [
        hmean(c, eps[2:, 1:-1, 1:-1]),
        hmean(c, eps[:-2, 1:-1, 1:-1]),
        hmean(c, eps[1:-1, 2:, 1:-1]),
        hmean(c, eps[1:-1, :-2, 1:-1]),
        hmean(c, eps[1:-1, 1:-1, 2:]),
        hmean(c, eps[1:-1, 1:-1, :-2]),
    ]
    den = sum(f)
    N = eps.shape[0]
    idx = np.arange(1, N - 1)
    I, J, K = np.meshgrid(idx, idx, idx, indexing="ij")
    parity = (I + J + K) % 2
    for _ in range(n_iter):
        for colour in (0, 1):
            num = (
                f[0] * V[2:, 1:-1, 1:-1]
                + f[1] * V[:-2, 1:-1, 1:-1]
                + f[2] * V[1:-1, 2:, 1:-1]
                + f[3] * V[1:-1, :-2, 1:-1]
                + f[4] * V[1:-1, 1:-1, 2:]
                + f[5] * V[1:-1, 1:-1, :-2]
            )
            update = (1.0 - omega) * V[1:-1, 1:-1, 1:-1] + omega * num / den
            mask = parity == colour
            V[1:-1, 1:-1, 1:-1][mask] = update[mask]
    return V


def dielectric_sphere_field(N, L, eps_r, a=1.0, n_iter=500):
    """Interior field of a dielectric sphere in a unit applied field.

    Builds the permittivity grid (eps_r inside radius a, 1 outside) in a
    box of side L with Dirichlet V = -z on the boundary (uniform applied
    field E0 = 1), relaxes with sor_epsilon, and returns the z-field at
    the center plus its spread over a 3x3x3-point interior sample — the
    uniformity that eq-fm-sphere predicts.

    Parameters
    ----------
    N : int
        Grid points per side.
    L : float
        Box side, in units of the sphere radius.
    eps_r : float
        Relative permittivity of the sphere.
    a : float
        Sphere radius.
    n_iter : int
        Red–black sweeps.

    Returns
    -------
    tuple of float
        ``(E_center, E_spread)``: the central E_z (units of E0) and the
        standard deviation over the interior sample.
    """
    h = L / N
    x = np.linspace(-L / 2, L / 2, N)
    X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
    R = np.sqrt(X**2 + Y**2 + Z**2)
    eps = np.where(R <= a, eps_r, 1.0)
    V = -Z.copy()
    sor_epsilon(eps, V, n_iter)
    i0 = N // 2
    d = max(2, int(0.25 * a / h))
    samples = []
    for di in (-d, 0, d):
        for dj in (-d, 0, d):
            for dk in (-d, 0, d):
                i, j, k = i0 + di, i0 + dj, i0 + dk
                samples.append(-(V[i, j, k + 1] - V[i, j, k - 1]) / (2 * h))
    return float(np.mean(samples)), float(np.std(samples))

## Exercise 1 — Bound charge is real: the polarized sphere

{eq}`eq-fm-bound` says a uniformly polarized sphere carries the surface
charge $\sigma_b = P\cos\theta$ (positive cap where $\mathbf P$ exits,
negative where it enters) and no volume charge, and that this charge —
treated as ordinary charge in Coulomb's law — reproduces the material's
entire effect. The exact consequence is famous: the interior field is
*uniform*, $\mathbf E = -\mathbf P/(3\varepsilon_0)$.

**Part a)** Discretize the unit sphere's surface on a $200\times200$
$(\theta, \phi)$ grid, load each patch with $\sigma_b = P\cos\theta$
(take $P/\varepsilon_0 = 1$ so fields are in units of
$P/\varepsilon_0$), and sum the Coulomb field of every patch at the
center: by symmetry only $E_z$ survives, and each patch of area $dA$ a
distance $1$ away contributes $-\sigma_b\,dA\,\cos\theta / (4\pi)$
along $z$. Verify the sum gives $E_z = -1/3$ (in these units) to
`rtol=1e-2` — the grid's residual — and verify the total bound charge
sums to zero (`atol=1e-12`): the sphere is neutral, merely rearranged.

**Part b)** Verify the two hemispheres carry equal and opposite charge
$\pm\pi P a^2$ (`rtol=1e-3` against $\pi$): the cap-and-belt pattern
integrates to the *dipole* the far field sees, $p = \tfrac43\pi a^3 P$
— check that too, by summing $z$-weighted surface charge
(`rtol=1e-3`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    E_center,
    -1.0 / 3.0,
    "summing the sigma_b = P cos(theta) surface patches gives the "
    "uniform interior field -P/(3 eps0): bound charge is real charge",
    rtol=1e-2,
)
validate.check(
    abs(Q_total) < 1e-12,
    "and it sums to zero: the sphere is neutral, its charge merely " "rearranged",
    f"ΣQ = {Q_total:.1e}",
)
validate.close(
    np.array([Q_north, p_dipole]),
    np.array([np.pi, 4.0 * np.pi / 3.0]),
    "each hemisphere carries ±π P a², assembling the dipole moment "
    "(4π/3) a³ P the far field sees",
    rtol=1e-3,
)

## Exercise 2 — The dielectric sphere, solved by relaxation

{eq}`eq-fm-sphere` is the subject's exact showpiece; here it becomes
the benchmark for a *numerical* method that generalizes far beyond
spheres. Discretizing {eq}`eq-fm-eps` on a grid turns each site into a
flux balance with its six neighbours, the face permittivities entering
as *harmonic means* (that choice, not the arithmetic mean, is what
keeps $D_\perp$ continuous across the jump); red–black over-relaxation
— the checkerboard variant of [§3.4](laplace-poisson.ipynb)'s SOR,
which stays stable at $\omega$ near $2$ because each update uses
already-updated neighbours of the other colour — then relaxes the
potential. `sor_epsilon` implements the sweep; **write this one
yourself** — the implementation is the lesson.

**Part a)** Solve for a sphere of $\varepsilon_r = 4$ and radius
$a = 1$ in a box of side $L = 8a$ on a $96^3$ grid with Dirichlet
$V = -E_0 z$ on the boundary ($E_0 = 1$), $500$ red–black sweeps at
$\omega = 1.95$, via `dielectric_sphere_field`. Verify the interior
field is *uniform* (spread below $2\times10^{-3}$ over a 27-point
interior sample) and equals $3/(\varepsilon_r + 2) = 0.5$ to
`rtol=5e-2`.

**Part b)** Face the systematics instead of hiding them. The few-percent
residual has two named sources: the finite box (the sphere's induced
dipole falls only as $(a/r)^3$, and the boundary clamps it to the
applied value early) and the staircase surface (the grid resolves the
sphere to $\pm h$). Verify the box-size claim directly: re-solve in the
*smaller* box $L = 4a$ ($64^3$, $400$ sweeps) and verify its error
against the exact $0.5$ is *larger* than the $L = 8a$ run's — the
systematic shrinks in the direction the physics predicts. Numerical
answers come with error budgets, or they come with asterisks.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    spread_8 < 2e-3,
    "the interior field is uniform across a 27-point sample: the "
    "hallmark of the exact solution, reproduced",
    f"spread {spread_8:.1e}",
)
validate.close(
    E_in_8,
    E_exact,
    "and equals 3 E0/(eps_r + 2) = 0.5 to the solver's few-percent " "systematics",
    rtol=5e-2,
)
validate.check(
    abs(E_in_4 - E_exact) > abs(E_in_8 - E_exact),
    "halving the box enlarges the error, as the truncated induced "
    "dipole predicts: the systematic has a name and a direction",
    f"|err|: {abs(E_in_4 - E_exact):.4f} (L=4a) vs "
    f"{abs(E_in_8 - E_exact):.4f} (L=8a)",
)

## Exercise 3 — Clausius–Mossotti: one atom prices the gas

{eq}`eq-fm-cm` turns a single molecular number into a bulk material
property. Argon's measured polarizability volume is $\alpha' =
1.6411\ \mathrm{\AA^3}$; at $0\,^\circ\mathrm{C}$ and $1$ atm the gas
has number density $n = p/k_BT$.

**Part a)** Evaluate the Clausius–Mossotti prediction
$\varepsilon_r = (1 + 2x)/(1 - x)$ with $x = 4\pi n\alpha'/3$ (from
`scipy.constants` values of $k_B$ and the atmosphere) and verify the
predicted susceptibility $\chi_e = \varepsilon_r - 1$ matches argon's
measured $\varepsilon_r = 1.000516$ at `rtol=0.1`: one tabulated atom,
the bulk gas to ten percent (the residual is the polarizability
value's own frequency convention and the non-ideality of the gas, not
the local-field logic).

**Part b)** Verify the dilute hierarchy: at this density the
local-field correction is tiny — $\chi_e$ from Clausius–Mossotti and
from the naive $\chi_e = 4\pi n\alpha'$ (no local field) agree to
`rtol=1e-3` — and verify where that stops being true: the polarization
catastrophe $x \to 1$ sits at $n_{\rm crit} = 3/(4\pi\alpha')$,
about $1.5\times10^{29}\ \mathrm{m^{-3}}$ — verify it exceeds the STP
density by a factor above $5000$ (gases are safe) yet lies within a
factor of $6$ of a typical liquid's $\sim 2.5\times10^{28}$ (dense
matter is not, which is why water's $\varepsilon_r \approx 80$ needs
more than this formula, and why ferroelectrics exist).

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    chi_cm,
    EPSR_MEASURED - 1.0,
    "one tabulated atomic polarizability prices argon's bulk "
    "susceptibility to ten percent",
    rtol=0.1,
)
validate.close(
    chi_cm,
    chi_naive,
    "at gas density the local-field correction is negligible: "
    "Clausius–Mossotti and the naive count agree to a part in a "
    "thousand",
    rtol=1e-3,
)
validate.check(
    n_crit / n_gas > 5000.0 and n_crit / N_LIQUID < 6.0,
    "but the polarization catastrophe sits thousands of times beyond "
    "gas density and within one order of a liquid's: dense matter "
    "outgrows the formula",
    f"n_crit/n_gas = {n_crit / n_gas:.0f}, n_crit/n_liq = " f"{n_crit / N_LIQUID:.1f}",
)

## Exercise 4 — The magnetized cylinder is a solenoid

The magnetic bookkeeping mirrors the electric: {eq}`eq-fm-bound`'s
analogue replaces a uniformly magnetized cylinder by its bound surface
current $K_b = M$ — a sheet current identical to a tightly wound
solenoid's. The claim is checkable to high precision because both
sides are computable: the cylinder's on-axis field from stacking
current loops, and the closed form {eq}`eq-fm-cylinder`.

**Part a)** For a cylinder of radius $R = 0.5$ and half-length
$\ell = 1$ (lengths in units of $\ell$, fields in units of $\mu_0 M$):
stack $2000$ circular current loops of strength $M\,dz$ (the on-axis
loop field $B_z = \mu_0 I R^2 / [2(R^2 + z^2)^{3/2}]$ from
[§3.6](magnetostatics.ipynb)) and verify the center field matches
{eq}`eq-fm-cylinder` at $z = 0$, which reduces to $\mu_0 M\,\ell/
\sqrt{\ell^2 + R^2} = 0.8944\,\mu_0 M$, to `rtol=1e-3`.

**Part b)** Verify the full on-axis profile: evaluate the loop-stack
sum at the eleven positions $z = 0, \pm0.25, \pm0.5, \pm0.75, \pm1,
\pm1.5$ and verify every point matches {eq}`eq-fm-cylinder` to
`rtol=1e-3` — including the ends, where the field has dropped to
roughly half its central value (the classic solenoid-end factor), and
outside, where the equivalent bar magnet's field is decaying toward
its dipole tail.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    b_stack[0],
    b_exact[0],
    "2000 stacked bound-current loops reproduce the solenoid closed "
    "form at the center: K_b = M is the whole magnet",
    rtol=1e-3,
)
validate.close(
    b_stack,
    b_exact,
    "and along the full axis, ends and dipole tail included",
    rtol=1e-3,
)
validate.check(
    0.45 < b_stack[7] / b_stack[0] < 0.55,
    "with the field at the end face near half its central value: the "
    "classic solenoid-end factor",
    f"ratio {b_stack[7] / b_stack[0]:.3f}",
)

## Exercise 5 — Hysteresis: when the response becomes memory

The mean-field equation {eq}`eq-fm-meanfield` is where matter's
response stops being a passive coefficient. Above $T_c = 1$ it has one
solution and the material is a paramagnet; below, at small $|h|$, it
has three, the outer two stable — and a field sweep drags the system
along whichever stable branch it currently occupies until that branch
*disappears*, at which point the magnetization jumps. Solving the
equation by fixed-point iteration seeded with the *previous* state is
not a numerical shortcut; it is the physics of memory, implemented.

**Part a)** At $T = 0.7$, sweep $h$ from $-0.3$ to $+0.3$ and back in
$121$ steps each way, at each step iterating $m \mapsto
\tanh((m + h)/T)$ two thousand times from the previous $m$. Verify the
loop's three signatures: the remanence $|m(h = 0)|$ on each branch
equals the *spontaneous* magnetization (the $h = 0$ solution of
{eq}`eq-fm-meanfield`, $m_s = 0.8286$ at this $T$) to `rtol=1e-3` —
two different computations, one number; the coercive field (where the
branch's magnetization flips sign) lands in $[0.10, 0.13]$; and the
enclosed loop area $\oint m\,dh$ is positive and exceeds $0.3$ — work
dissipated per cycle, the price of rewriting a memory.

**Part b)** Verify the loop *closes* above the transition: repeat the
sweep at $T = 1.5$ and verify the up- and down-branches coincide
everywhere (maximum difference below $10^{-6}$) — a paramagnet
responds but does not remember. Then verify the mean-field
susceptibility diverges on approach: compute $\chi = dm/dh|_{h=0}$
numerically (centered difference with $h = \pm10^{-4}$) at $T = 1.5,
2, 3$, fit $1/\chi$ against $T$ (`numpy.polyfit`, degree 1), and
verify the line crosses zero at $T_c = 1$ within `rtol=2e-2`:
Curie–Weiss, measured from the response — the same mean-field critical
point [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)
analyses from the statistical side.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    remanence,
    m_spont,
    "the remanence equals the spontaneous magnetization: two "
    "computations (a sweep's h=0 crossing, the h=0 fixed point), one "
    "number",
    rtol=1e-3,
)
validate.check(
    0.10 < h_coercive < 0.13,
    "the branch survives until its coercive field near 0.115, then "
    "jumps: the memory has a measurable price of erasure",
    f"h_c = {h_coercive:.4f}",
)
validate.check(
    loop_area > 0.3,
    "the enclosed loop area — work dissipated per cycle — is finite "
    "and positive: rewriting a magnet costs energy",
    f"area = {loop_area:.4f}",
)
validate.check(
    hot_gap < 1e-6,
    "above T_c the branches coincide to 1e-6: a paramagnet responds "
    "but does not remember",
    f"gap = {hot_gap:.1e}",
)
validate.close(
    Tc_fit,
    1.0,
    "and the inverse susceptibility's straight line points at T_c = 1: "
    "Curie–Weiss, measured from the response side",
    rtol=2e-2,
)

## Notebook summary

- Bound charge behaved as advertised: summing $\sigma_b = P\cos\theta$
  over the polarized sphere's surface gave the uniform interior field
  $-P/(3\varepsilon_0)$ to the grid's percent, with zero net charge,
  $\pm\pi Pa^2$ per hemisphere, and the dipole moment
  $\tfrac43\pi a^3P$ the far field sees.
- The permittivity-jump relaxation solver (harmonic-mean face
  coefficients, red–black SOR) reproduced the dielectric sphere's
  exact $3E_0/(\varepsilon_r + 2)$ with a uniform interior
  (spread $5\times10^{-4}$), and its few-percent residual *shrank when
  the box grew* — a systematic with a name and a direction, not an
  asterisk.
- Clausius–Mossotti turned argon's tabulated
  $\alpha' = 1.6411\ \mathrm{\AA^3}$ into the measured bulk
  $\varepsilon_r = 1.000516$ within ten percent, with the local-field
  correction negligible at gas density and the polarization
  catastrophe correctly located near liquid densities.
- The magnetized cylinder *was* its bound-current solenoid: 2000
  stacked loops matched the closed form along the whole axis to
  $10^{-3}$, with the end-face field at half the central value.
- Below $T_c$ the mean-field self-consistency became memory: remanence
  equal to the spontaneous magnetization to $10^{-3}$, coercive field
  $0.115$, dissipative loop area $0.38$ — and above $T_c$ the loop
  closed to $10^{-6}$ while the inverse susceptibility's Curie–Weiss
  line pointed at $T_c = 1$ to two percent.

## Outlook

- **Frequency response.** Let the applied field oscillate and
  $\varepsilon$ becomes $\varepsilon(\omega)$, complex and causal: the
  dispersion and absorption of [§3.8](maxwell-waves.ipynb)'s waves in
  real media, with the Kramers–Kronig relations enforcing causality —
  machinery this course meets from the response side in
  [§7.25](../07-quantum-statistical-mechanics/linear-response-kubo.ipynb).
- **Effective-medium theory.** The dielectric sphere's
  $3/(\varepsilon_r+2)$ is the seed of Maxwell Garnett and Bruggeman
  mixing rules for composites — the same geometry, averaged over
  inclusions.
- **Real ferroelectrics and ferromagnets.** Domains, domain walls, and
  the pinning that turns the clean mean-field loop into the jagged
  Barkhausen reality; the statistical side of the transition is
  [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)'s
  subject, and the microscopic origin of $\mathbf M$ itself is spin —
  [§6.18](../06-quantum-mechanics/spin-magnetic.ipynb).
- **Microscopic polarizability.** The $\alpha$ fed to
  Clausius–Mossotti is quantum mechanics' to compute: perturbation
  theory delivers it in
  [§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb),
  closing the loop from atom to capacitor.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()